# Tarea 1: Predicción de resultados del fútbol uruguayo

Este notebook tiene el objetivo de reflejar todos los pasos que se consideraron necesarios para satisfacer los requerimientos de la Tarea 1 del laboratorio de Apendizajes Automáticos.

Para la correcta verificación de cada uno de los pasos indicados en el notebook, se asume de antemano que se cuenta con la instalación de Python 3.12.x en su máquina local.

In [3]:
# Instalación de las librerías Python necesarias

%pip install pandas numpy matplotlib scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\lucas\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import numpy as np
import sklearn

--- 
### 1. Carga del Dataset y Descripción de Atributos

In [5]:
DATASET_FILE = "./futbol_uruguayo.csv" 

dataset = pd.read_csv(DATASET_FILE)

print(f"Cantidad total de instancias: ", dataset.shape[0])
print(f"Cantidad total de atributos: ", dataset.shape[1])
dataset.head()

Cantidad total de instancias:  15207
Cantidad total de atributos:  17


,home,away,date,gh,ga,full_time,competition,home_ident,away_ident,home_country,away_country,home_code,away_code,home_continent,away_continent,continent,level
0,Bella Vista,Defensor Sporting,1932-03-05,1.0,2.0,F,uruguay,Bella Vista (Uruguay),Defensor Sporting (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national
1,CA Penarol,River Plate,1932-03-05,1.0,1.0,F,uruguay,CA Penarol (Uruguay),River Plate (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national
2,Montevideo Wanderers,Racing Club,1932-03-05,3.0,0.0,F,uruguay,Montevideo Wanderers (Uruguay),Racing Club (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national
3,Central Espanol,Rampla Juniors Futbol Club,1932-03-05,1.0,0.0,F,uruguay,Central Espanol (Uruguay),Rampla Juniors Futbol Club (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national
4,Nacional,Institucion Atletica Sud America,1932-03-05,2.0,0.0,F,uruguay,Nacional (Uruguay),Institucion Atletica Sud America (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national


#### Descripción de los atributos:

| Atributo | Descripción |
| :--- | :--- |
| **`home`** | Nombre del equipo local (no necesariamente único) |
| **`away`** | Nombre del equipo visitante (no necesariamente único) |
| **`date`** | Fecha del partido |
| **`gh`** | Goles del equipo local (incluyendo tiempo extra y penales) |
| **`ga`** | Goles del equipo visitante (incluyendo tiempo extra y penales) |
| **`full_time`** | "F"=el partido terminó en 90', "E"=tiempo extra, "P"=penales |
| **`competition`** | Nombre del país de la liga o nombre de la competición int. |
| **`home_ident`** | Identificador único del equipo local |
| **`away_ident`** | Identificador único del equipo visitante |
| **`home_country`** | País del equipo local |
| **`away_country`** | País del equipo visitante |
| **`home_code`** | Código de país del equipo local |
| **`away_code`** | Código de país del equipo visitante |
| **`home_continent`** | Continente del equipo local |
| **`away_continent`** | Continente del equipo visitante |
| **`continent`** | Continente de la competición |
| **`level`** | "national"= liga local, "international"= copa internacional |

--- 
### 2. Definición de la Variable Objetivo (`ganador`)

El objetivo del modelo es predecir el resultado final de un partido de fútbol, clasificándolo en una de tres categorías posibles: victoria local, victoria visitante o empate.

Dado que el dataset original no incluye directamente una columna de resultado, se deduce la variable objetivo **`ganador`** mediante la comparación de los goles anotados por el equipo local (`gh`) y el visitante (`ga`):

* Si $\text{gh} > \text{ga} \implies$ **`L`**
* Si $\text{ga} > \text{gh} \implies$ **`V`**
* Si $\text{gh} == \text{ga} \implies$ **`E`**

In [6]:
dataset["ganador"] = np.select(
    [
        dataset["gh"] > dataset["ga"],     
        dataset["gh"] < dataset["ga"],
        dataset["gh"] == dataset["ga"],
    ],
    [
        "L",
        "V",
        "E",
    ],
    default="sin_dato",
)

dataset.head()

,home,away,date,gh,ga,full_time,competition,home_ident,away_ident,home_country,away_country,home_code,away_code,home_continent,away_continent,continent,level,ganador
0,Bella Vista,Defensor Sporting,1932-03-05,1.0,2.0,F,uruguay,Bella Vista (Uruguay),Defensor Sporting (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national,V
1,CA Penarol,River Plate,1932-03-05,1.0,1.0,F,uruguay,CA Penarol (Uruguay),River Plate (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national,E
2,Montevideo Wanderers,Racing Club,1932-03-05,3.0,0.0,F,uruguay,Montevideo Wanderers (Uruguay),Racing Club (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national,L
3,Central Espanol,Rampla Juniors Futbol Club,1932-03-05,1.0,0.0,F,uruguay,Central Espanol (Uruguay),Rampla Juniors Futbol Club (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national,L
4,Nacional,Institucion Atletica Sud America,1932-03-05,2.0,0.0,F,uruguay,Nacional (Uruguay),Institucion Atletica Sud America (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national,L


Concretado este paso, se observa que la información que proveen los atributos "gh" y "ga" ya se ve contemplada por la variable objetivo. Por lo tanto, su presencia en el dataset no agrega significancia al entrenamiento del modelo y se deben remover las columnas correspondientes a dichos atributos.

--- 
### 3. Selección de Atributos

En esta etapa del preprocesamiento se realiza una selección de los atributos del dataset con el objetivo de maximizar la capacidad de aprendizaje del modelo en la clasificación de las instancias. Para ello, se examina cada atributo de forma individual para evaluar si contribuye de manera significativa al entrenamiento o si, dadas sus características, puede ser removido sin perjudicar el desempeño del modelo.

Se evalúan las siguientes condiciones de los atributos:

* **Atributos constantes:** No contribuyen al aprendizaje del modelo debido a que mantienen el mismo valor para todas las instancias del dataset (varianza cero).
* **Atributos redundantes:** Representan valores equivalentes dentro del dataset, por lo que basta con conservar uno de ellos.

#### A. Detección de Atributos Constantes

A modo informativo, se evalúan los atributos constantes en el dataset contando la cantidad de instancias con valores diferentes para cada atributo en el dataset. Luego, se imprime el nombre de aquellos atributos para los que existe un único valor equivalente para todas las instancias en el dataset.

In [7]:
# Detección de Atributos Constantes
constantes = dataset.nunique(dropna=False)[dataset.nunique(dropna=False) == 1]
print("Atributos constantes detectados:")
print(constantes)

# Eliminar todas las columnas con un solo valor único
dataset = dataset.loc[:, dataset.nunique() > 1]

Atributos constantes detectados:
competition       1
home_country      1
away_country      1
home_code         1
away_code         1
home_continent    1
away_continent    1
continent         1
level             1
dtype: int64


Los atributos constantes son los siguientes:
* `competition`, `home_country`, `away_country` (Todos refieren a Uruguay).
* `home_code`, `away_code` (Código constante `UY`).
* `home_continent`, `away_continent`, `continent` (Todos refieren a `South America`).
* `level` (Constante con el valor `national`).

Estos atributos se excluyen para que no formen parte del entrenamiento del modelo.

#### B. Detección de Atributos Redundantes

In [25]:
# Verificar cuántos identificadores tiene cada nombre de equipo, y viceversa
home_name_to_id = dataset.groupby("home")["home_ident"].nunique(dropna=False)
home_id_to_name = dataset.groupby("home_ident")["home"].nunique(dropna=False)

away_name_to_id = dataset.groupby("away")["away_ident"].nunique(dropna=False)
away_id_to_name = dataset.groupby("away_ident")["away"].nunique(dropna=False)

print(f"Cada valor de home se corresponde a un solo valor de home_ident, y viceversa: ", (home_name_to_id == 1).all()  & (home_id_to_name == 1).all())
print(f"Cada valor de away se corresponde a un solo valor de away_ident, y viceversa: ", (away_name_to_id == 1).all()  & (away_id_to_name == 1).all())

dataset = dataset.drop(columns=['home_ident', 'away_ident'])

Cada valor de home se corresponde a un solo valor de home_ident, y viceversa:  True
Cada valor de away se corresponde a un solo valor de away_ident, y viceversa:  True


Existe una correspondencia 1:1, mantener ambas variables introduciría información redundante. Elegimos quedarnos con `home` y `away` y se descartan sus identificadores (`home_ident` y `away_ident`).

**Obervación**: La aplicación de este método "a mano" para identificar a los atributos redundantes es más que suficiente para el dataset con el que estamos trabajando, sin embargo, si se tratara de un dataset con una cantidad más grande de atributos entonces sería necesario automatizar de alguna forma la identificación de los atributos redundantes mediante la medición de la correlación de atributos.

#### C. Descomposición del atributo `date`

El atributo `date` se divide en los atributos `day`, `month` y `year` para representar sus componentes por separado. Esta transformación facilita que el modelo identifique patrones relacionados con el momento del año en que se disputó el partido, como diferencias entre meses, temporadas o períodos históricos. Además, evita tratar cada fecha completa como un valor independiente, lo que podría dificultar el aprendizaje. Una vez extraídos estos componentes, se elimina el atributo `date` original para evitar mantener información redundante.

In [15]:
dataset["date"] = pd.to_datetime(dataset["date"])
dataset["day_of_week"] = dataset["date"].dt.day_name()
dataset["month"] = dataset["date"].dt.month
dataset['year'] = dataset['date'].dt.year

---
### 4. Análisis de Balance y Estratificación:

In [16]:
# Cálculo de la dispersión de instancias respecto a las clases
porcentajes = dataset["ganador"].value_counts(normalize=True).mul(100).round(2)

instancias_L = float(porcentajes.iloc[0])
instancias_V = float(porcentajes.iloc[1])
instancias_E = float(porcentajes.iloc[2])

print(f"Porcentaje de instancias correspondientes a la clase L: ", instancias_L)
print(f"Porcentaje de instancias correspondientes a la clase V: ", instancias_V)
print(f"Porcentaje de instancias correspondientes a la clase E: ", instancias_E)

Porcentaje de instancias correspondientes a la clase L:  44.35
Porcentaje de instancias correspondientes a la clase V:  28.2
Porcentaje de instancias correspondientes a la clase E:  27.44


La distribución de la clase objetivo muestra las siguientes proporciones:
* **`L`**: **44.35%**
* **`V`**: **28.20%**
* **`E`**: **27.44%**

---
### 5. División del Conjunto de Datos

Se debe dividir el conjunto dataset de la siguiente forma:

Conjunto de entrenamiento los partidos jugados hasta el año 2023 inclusive y como conjunto de evaluación los partidos jugados en 2024 y 2025.


In [30]:
dataset = dataset.sort_values(by='date').reset_index(drop=True)

# Instancias con partidos jugados hasta el año 2023 inclusive
df_entrenamiento = dataset[dataset['year'] <= 2023]
X_train = df_entrenamiento.drop(columns=['date','ganador', 'gh', 'ga', 'year']) # ACLARAR LA ELIMINACION DE YEAR
Y_train = df_entrenamiento['ganador']

df_evaluacion = dataset[dataset['year'] > 2023]
X_test = df_evaluacion.drop(columns=['date', 'ganador', 'gh', 'ga', 'year'])
Y_test = df_evaluacion['ganador']

print(f"Dimensiones de X_train (Entrenamiento): {X_train.shape}")
print(f"Dimensiones de X_test (Evaluación):    {X_test.shape}")

Dimensiones de X_train (Entrenamiento): (14734, 5)
Dimensiones de X_test (Evaluación):    (473, 5)


In [34]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer


numeric_features = ['month']
categorical_features = ['home', 'away', 'full_time', 'day_of_week']

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(fill_value='missing', strategy='constant')),
    ('onehot', OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False
    ))],
)

preprocessing = ColumnTransformer(
    transformers=[
        ('numeric', numeric_pipeline, numeric_features),
        ('categorical', categorical_pipeline, categorical_features),
    ],
)


pipeline = Pipeline([
    ('preprocessing', preprocessing),
    ('model', DecisionTreeClassifier(random_state=62))
])

X_train.head()

,home,away,full_time,day_of_week,month
0,Bella Vista,Defensor Sporting,F,Saturday,3
1,CA Penarol,River Plate,F,Saturday,3
2,Montevideo Wanderers,Racing Club,F,Saturday,3
3,Central Espanol,Rampla Juniors Futbol Club,F,Saturday,3
4,Nacional,Institucion Atletica Sud America,F,Saturday,3


In [35]:
param_grid = {
    # Preprocesamiento numérico
    'preprocessing__numeric__imputer__strategy': [
        'mean',
        'median'
    ],
    # Árbol de decisión
    'model__max_depth': [
        3,
        6,
        10,
        15,
        None
    ],
    'model__criterion':[
        'gini','entropy'
    ],
    # Minima ganancia permitida para un atributo
    'model__min_impurity_decrease': [
        0.0,
        0.001,
        0.005,
        0.01,
        0.05
    ]
}

In [36]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

cv_temporal = TimeSeriesSplit(n_splits=10)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='accuracy',
    cv=cv_temporal,
    n_jobs=-1,
    refit=True
)

# GridSearchCV realiza internamente la validación cruzada
grid_search.fit(X_train, Y_train)


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=62))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__criterion': ['gini', 'entropy'], 'model__max_depth': [3, 6, ...], 'model__min_impurity_decrease': [0.0, 0.001, ...], 'preprocessing__numeric__imputer__strategy': ['mean', 'median']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",TimeSeriesSpl...est_size=None)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelco

In [38]:
import pandas as pd

resultados_gs = pd.json_normalize(grid_search.best_params_)
resultados_gs.columns = [col.replace('preprocessing__', '').replace('model__', '') for col in resultados_gs.columns]
resultados_gs['accuracy_cv'] = grid_search.best_score_

test_accuracy = grid_search.score(X_test, Y_test)

resultados_gs['accuracy_test'] = test_accuracy

display(resultados_gs)

,criterion,max_depth,min_impurity_decrease,numeric__imputer__strategy,accuracy_cv,accuracy_test
0,entropy,10,0.001,mean,0.480807,0.479915
